In [1]:
import pandas as pd
from pinecone import Pinecone
from dotenv import load_dotenv
import os
from tqdm import tqdm
import numpy as np
import ast

In [2]:
tqdm.pandas()

In [3]:
load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [4]:
index = pc.Index(host=os.getenv("INDEX_HOST"))
os.getenv("INDEX_HOST")

'https://dinov2base-hq9ybxr.svc.aped-4627-b74a.pinecone.io'

In [ ]:
df=pd.read_csv("../data/bw_vectors.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 616 entries, 0 to 615
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   image_paths  616 non-null    object
 1   vectors      616 non-null    object
dtypes: object(2)
memory usage: 9.8+ KB


In [7]:
df.dropna(subset="vectors", inplace=True)

In [8]:
df.fillna("None", inplace=True)

In [9]:
def normalize_vector(vector):
    vector = np.array(ast.literal_eval(vector), dtype=float)
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

df["vectors"] = df["vectors"].apply(normalize_vector)

In [10]:
vector = [
    (str(row["image_paths"]), np.array(row["vectors"], dtype=np.float32).tolist())
    for row in df.to_dict(orient="records")
]

In [11]:
vector[0]

('C:/Workspace/Matching Products/data/bw_design_clusters\\0006\\Screenshot 2025-05-19 115739.png',
 [-0.014454967342317104,
  -0.024602100253105164,
  0.04690379649400711,
  -0.05325515568256378,
  0.00040189645369537175,
  -0.0007515266770496964,
  -0.0527285635471344,
  -0.02331037074327469,
  0.004838787950575352,
  0.10324815660715103,
  -0.07421927154064178,
  0.06061634421348572,
  -0.033569641411304474,
  -0.0707402154803276,
  -0.0568530410528183,
  0.04152444377541542,
  -0.02639157697558403,
  0.00017341178318019956,
  -0.05305967479944229,
  -0.01293554063886404,
  -0.08605765551328659,
  0.05108477175235748,
  -0.02280876785516739,
  -0.03776342421770096,
  -0.028201643377542496,
  -0.0858406350016594,
  -0.049073148518800735,
  -0.010613735765218735,
  -0.04273994266986847,
  -0.04629659652709961,
  -0.021872252225875854,
  0.07470044493675232,
  -0.04767703637480736,
  0.10528136789798737,
  0.010497690178453922,
  0.0001885186939034611,
  0.08331996202468872,
  0.0838195

In [12]:
batch_size = 100

for i in tqdm(range(0, len(vector), batch_size)):
    batch = vector[i:i+batch_size]
    index.upsert(vectors=batch, namespace="bw")

100%|██████████| 7/7 [00:07<00:00,  1.04s/it]
